In [ ]:
import pyodbc
import pandas as pd

conn = pyodbc.connect(
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=LAPTOP-8TGEEHVT\TEW_SQLEXPRESS;"
    "DATABASE=DataWarehouse;"
    "Trusted_Connection=yes;"
)

print("Connected to SQL Server successfully!")


<>:7: SyntaxWarning: invalid escape sequence '\T'
<>:7: SyntaxWarning: invalid escape sequence '\T'
C:\Users\MIT\AppData\Local\Temp\ipykernel_31576\827529968.py:7: SyntaxWarning: invalid escape sequence '\T'
  "SERVER=LAPTOP-8TGEEHVT\TEW_SQLEXPRESS;"


Connected to SQL Server successfully!


In [2]:
query = """
SELECT
    YEAR(order_date) AS sales_year,
    MONTH(order_date) AS sales_month,
    country,
    SUM(sales_amount) AS monthly_sales
FROM gold.vw_sales_analytics
WHERE country IS NOT NULL
AND order_date IS NOT NULL
GROUP BY
    YEAR(order_date),
    MONTH(order_date),
    country
ORDER BY
    sales_year,
    sales_month,
    country;
"""

df = pd.read_sql(query, conn)
df.head()


C:\Users\MIT\AppData\Local\Temp\ipykernel_31576\2732256868.py:20: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


,sales_year,sales_month,country,monthly_sales
0,2010,12,Australia,20909
1,2010,12,Canada,3578
2,2010,12,France,3400
3,2010,12,United Kingdom,699
4,2010,12,United States,14833


In [3]:
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 242 entries, 0 to 241
Data columns (total 4 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   sales_year     242 non-null    int64 
 1   sales_month    242 non-null    int64 
 2   country        242 non-null    object
 3   monthly_sales  242 non-null    int64 
dtypes: int64(3), object(1)
memory usage: 7.7+ KB


In [4]:
df_encoded = pd.get_dummies(df, columns=["country"], drop_first=True)
df_encoded.head()

,sales_year,sales_month,monthly_sales,country_Canada,country_France,country_Germany,country_United Kingdom,country_United States,country_n/a
0,2010,12,20909,False,False,False,False,False,False
1,2010,12,3578,True,False,False,False,False,False
2,2010,12,3400,False,True,False,False,False,False
3,2010,12,699,False,False,False,True,False,False
4,2010,12,14833,False,False,False,False,True,False


In [5]:
X = df_encoded.drop("monthly_sales", axis=1)
y = df_encoded["monthly_sales"]


In [6]:
split_year = df_encoded["sales_year"].quantile(0.8)

train_data = df_encoded[df_encoded["sales_year"] <= split_year]
test_data  = df_encoded[df_encoded["sales_year"] > split_year]

X_train = train_data.drop("monthly_sales", axis=1)
y_train = train_data["monthly_sales"]

X_test = test_data.drop("monthly_sales", axis=1)
y_test = test_data["monthly_sales"]


In [7]:
from sklearn.linear_model import LinearRegression

model = LinearRegression()
model.fit(X_train, y_train)

print("Model trained successfully!")

Model trained successfully!


In [9]:
coefficients = pd.DataFrame({
    "Feature": X_train.columns,
    "Coefficient": model.coef_
}).sort_values(by="Coefficient", ascending=False)

coefficients


,Feature,Coefficient
0,sales_year,64407.668260
1,sales_month,6408.460702
6,country_United States,2533.243243
5,country_United Kingdom,-153143.702703
4,country_Germany,-166868.324246
3,country_France,-173340.945946
2,country_Canada,-191442.486486
7,country_n/a,-284825.742357


In [ ]:
def decision_support(predicted_sales, historical_avg, country, global_avg):
    
    decisions = []

    if predicted_sales > historical_avg * 1.10:
        decisions.append("Increase inventory allocation")
    elif predicted_sales < historical_avg * 0.90:
        decisions.append("Reduce inventory to avoid overstock")

    if country == "United States":
        decisions.append("High priority market")
    elif predicted_sales < global_avg:
        decisions.append("Marketing intervention required")

    if predicted_sales > historical_avg:
        decisions.append("Positive revenue growth expected")
    else:
        decisions.append("Revenue risk – monitor closely")

    return decisions


In [14]:
def create_base_input(year, month, country, df_template):
    base = df_template.iloc[[0]].copy()

    for col in base.columns:
        if base[col].dtype == "bool":
            base[col] = False
        else:
            base[col] = 0

    base["sales_year"] = year
    base["sales_month"] = month

    country_col = f"country_{country}"
    if country_col in base.columns:
        base[country_col] = True

    return base


In [ ]:
base_input = create_base_input(
    year=2026,
    month=1,
    country="United States",
    df_template=X_train
)

predicted_sales = model.predict(base_input)[0]

what_if_sales = predicted_sales * 1.15

predicted_sales, what_if_sales


(np.float64(1116162.1948226243), np.float64(1283586.5240460178))

In [21]:
base_input = create_base_input(
    year=2026,
    month=1,
    country="Canada",
    df_template=X_train
)

predicted_sales = model.predict(base_input)[0]
what_if_sales = predicted_sales * 0.80

predicted_sales, what_if_sales


(np.float64(922186.4650928974), np.float64(737749.1720743179))

In [22]:
low_month = create_base_input(2026, 2, "Australia", X_train)
peak_month = create_base_input(2026, 7, "Australia", X_train)

low_pred = model.predict(low_month)[0]
peak_pred = model.predict(peak_month)[0]

low_pred, peak_pred


(np.float64(1120037.4122810662), np.float64(1152079.7157895118))

In [ ]:
historical_avg_by_country = (
    df.groupby("country")["monthly_sales"]
      .mean()
      .to_dict()
)

historical_avg_by_country


{'Australia': 238424.86842105264,
 'Canada': 52045.60526315789,
 'France': 69505.86842105263,
 'Germany': 78218.0,
 'United Kingdom': 89185.42105263157,
 'United States': 241111.18421052632,
 'n/a': 15121.333333333334}

In [25]:
global_avg_sales = df["monthly_sales"].mean()
global_avg_sales


np.float64(121286.19008264462)

In [34]:
country = "United States"

decisions = decision_support(
    predicted_sales=what_if_sales,
    historical_avg=historical_avg_by_country[country],
    country=country,
    global_avg=global_avg_sales
)

decisions


['Increase inventory allocation',
 'High priority market',
 'Positive revenue growth expected']